In [1]:
import os
import sys

import pandas as pd
import numpy as np
import random
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from torch.utils.tensorboard import SummaryWriter

In [2]:
df = pd.read_csv("rfv_matrix.csv")

In [3]:
df[df.columns] = df[df.columns].astype(int)

In [4]:
df.columns

Index(['RFV1_Administrative Module', 'RFV1_Adverse Effects of Drugs',
       'RFV1_Cardiovascular System',
       'RFV1_Diagnostic Tests (e.g., labs, imaging)', 'RFV1_Digestive System',
       'RFV1_Diseases of the Blood and Blood-forming Organs',
       'RFV1_Diseases of the Cardiovascular System',
       'RFV1_Diseases of the Digestive System',
       'RFV1_Diseases of the Genitourinary System',
       'RFV1_Diseases of the Musculoskeletal System and Connective Tissue',
       'RFV1_Diseases of the Nervous System',
       'RFV1_Diseases of the Respiratory System',
       'RFV1_Diseases of the Skin and Subcutaneous Tissue',
       'RFV1_Ear and Mastoid Process',
       'RFV1_Endocrine, Nutritional, and Metabolic Diseases',
       'RFV1_Eye and Adnexa', 'RFV1_General Examinations',
       'RFV1_General Symptoms', 'RFV1_Genitourinary System',
       'RFV1_Infectious and Parasitic Diseases',
       'RFV1_Injuries by Body Site (Head)',
       'RFV1_Medications (Refills, injections, adjust

In [5]:
len(df.columns)

38

In [6]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# data generation
X = np.random.randint(0, 2, size=(5000000, 38))


In [7]:
X


array([[0, 1, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 1, 0, 1],
       [1, 1, 0, ..., 0, 1, 1],
       ...,
       [1, 1, 0, ..., 0, 1, 1],
       [1, 1, 0, ..., 0, 0, 1],
       [0, 1, 0, ..., 0, 0, 0]], shape=(5000000, 38), dtype=int32)

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,X,test_size=0.2,random_state=SEED)

In [10]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 1, 1]], shape=(4000000, 38), dtype=int32)

In [11]:
y_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 1, 1]], shape=(4000000, 38), dtype=int32)

In [12]:
X_train = torch.FloatTensor(X_train).cuda()
y_train = torch.LongTensor(y_train).cuda()
X_test = torch.FloatTensor(X_test).cuda()
y_test = torch.LongTensor(y_test).cuda()

In [13]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [14]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [15]:
from models.rfv_autoencoder import RFVAutoEncoder,RFVDecoder,RFVEncoder

In [16]:
rfv_encoder = RFVEncoder(input_dim=38,hidden_dim=[64,32,16],output_dim=8)
rfv_decoder = RFVDecoder(input_dim=8,hidden_dim=[16,32,64],output_dim=38)
rfv_auto_encoder = RFVAutoEncoder(encoder=rfv_encoder,decoder=rfv_decoder).cuda()

In [17]:
def count_parameters(model):
    params = [p.numel() for p in model.parameters() if p.requires_grad]
    for item in params:
        print(f"{item:>6}")
    print(f"______\n{sum(params):>6}")

In [18]:
count_parameters(rfv_auto_encoder)

  2432
    64
  2048
    32
   512
    16
   128
     8
   128
    16
   512
    32
  2048
    64
  2432
    38
______
 10510


In [19]:
EPOCHS = 10000
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(rfv_auto_encoder.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min')
writer = SummaryWriter()
os.makedirs("checkpoint",exist_ok=True)
os.makedirs("best_model",exist_ok=True)


In [ ]:
patience = 0
best_loss = float("inf")
for i in range(EPOCHS):
    rfv_auto_encoder.train()
    epoch_training_loss = 0
    val_loss = 0
    current_lr = optimizer.param_groups[0]["lr"]
    writer.add_scalar('lr',current_lr,i)
    for b, (X_tr,y_tr) in enumerate(train_dataloader):
        y_pred = rfv_auto_encoder(X_tr)
        loss = criterion(y_pred, y_tr.float())
        epoch_training_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    rfv_auto_encoder.eval()
    with torch.no_grad():
        for b, (X_te,y_te) in enumerate(test_dataloader):
            y_pred = rfv_auto_encoder(X_te)
            loss = criterion(y_pred, y_te.float())
            val_loss += loss.item()
    train_loss = epoch_training_loss/len(train_dataloader)
    val_loss = val_loss/len(test_dataloader)
    writer.add_scalar('training_loss',train_loss,i)
    writer.add_scalar('validation_loss',val_loss,i)
    scheduler.step(val_loss)
    print("------------------------------------------------------------------------")
    print("Epoch:",i)
    print("Training Loss:",train_loss)
    print("Validation Loss: :",val_loss)
    print("Patience: ", patience)
    print("Learning Rate: ", current_lr)
    print("-------------------------------------------------------------------------")
    if val_loss < best_loss:
        torch.save(rfv_auto_encoder.state_dict(),"best_model/rfv_autoencoder.pth")
        os.makedirs(f"best_model_checkpoint_{i}",exist_ok=True)
        torch.save(
            {
                "epoch": i,
                "model": rfv_auto_encoder.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
            },f"best_model_checkpoint_{i}/rfv_autoencoder.pth"
        )
        best_loss = val_loss
    if i % 10 == 0:
        os.makedirs(f"epoch_checkpoint_{i}",exist_ok=True)
        torch.save(
            {
                "epoch": i,
                "model": rfv_auto_encoder.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
            },f"epoch_checkpoint_{i}/rfv_autoencoder.pth"
        )
    if train_loss < val_loss:
        patience += 1
    elif train_loss > val_loss:
        patience = 0
    else:
        pass

    if patience >=5:
        print("Early stopping")
        break

writer.flush()